# 02 – SARIMA (Leitfaden Punkte 5–7)

Ziel: Ein saisonales ARIMA-Modell auf den Trainingsdaten identifizieren und anpassen, die 24 Testmonate prognostizieren und unter denselben Bedingungen wie Holt-Winters und die Seasonal-Naive-Baseline bewerten (gleicher Split, gleiche Metriken aus `src/evaluation.py`).

Regeln gegen Datenleckage:
- Die Abschnitte 1–6 verwenden **nur die Trainingsdaten**; die Testmenge wird dort gar nicht geladen (`train, _ = ...`).
- Das endgültige Modell wird in Abschnitt 6 **vor** der Prognose festgelegt.
- Die Testdaten werden erst in Abschnitt 7 einmalig verwendet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_umsatz
from src.evaluation import train_test_split
from src import arima
from src.plots import speichern

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, _ = train_test_split(s)
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")

## 1. Varianzstabilisierung (Box-Cox)

ARIMA setzt eine (schwach) stationäre Reihe voraus, also auch eine konstante Varianz (Vorlesung 3, Folie 5). Ist die Varianz offensichtlich nicht konstant, wird sie mit einer Box-Cox-Transformation stabilisiert (Vorlesung 2, Folien 32–34).

**Entscheidungsregel (vor der Analyse festgelegt):** Transformiert wird nur, wenn die Standardabweichung je Jahr deutlich mit dem Jahresmittelwert steigt (Vorgehen wie in der Beispiel-Hausarbeit, Kap. 3). Das per Software geschätzte λ wird ergänzend berichtet.

In [ ]:
tab = arima.jahres_streuung(train)
r = tab["mittelwert"].corr(tab["std"])
print(f"Vollständige Jahre: {tab.index[0]}–{tab.index[-1]} ({len(tab)} Jahre)")
print(f"Korrelation Jahresmittelwert – Standardabweichung: r = {r:.2f}")
tab.round(2)

In [ ]:
fig = arima.plot_jahres_streuung(tab)
speichern(fig, "08_jahresstreuung")

In [ ]:
lam, ki = arima.boxcox_lambda(train)
print(f"λ (Maximum Likelihood) = {lam:.2f}, 95%-Konfidenzintervall [{ki[0]:.2f}; {ki[1]:.2f}]")

**Ergebnis und Entscheidung:**
- Zwischen Jahresmittelwert und jährlicher Standardabweichung besteht praktisch kein Zusammenhang (r = 0,06).
- Die größte Streuung tritt bei niedrigem Niveau auf (1999: Mittelwert 87,06, Standardabweichung 7,53), die kleinste bei vergleichsweise hohem Niveau (2022: 95,50 bzw. 4,52).
- Die Varianz ist damit nicht offensichtlich niveauabhängig. Gemäß der festgelegten Regel wird **keine Box-Cox-Transformation** angewendet; modelliert wird die Originalreihe.
- Das per Maximum Likelihood geschätzte λ = −1,76 (95%-KI [−2,84; −0,70]) dient nicht als Entscheidungsgrundlage: Das Verfahren unterstellt unabhängige, identisch verteilte Beobachtungen, was bei einer Reihe mit Trend und Saison nicht erfüllt ist.
- Konsequenz: Die Prognosen müssen nicht zurücktransformiert werden, und AIC-Vergleiche erfolgen auf der Originalskala.

## 2. Differenzierung

Ziel ist eine stationäre Reihe: konstanter Mittelwert, konstante Varianz, Kovarianz nur vom zeitlichen Abstand abhängig (Vorlesung 3, Folie 5). Differenzen werden mit dem Backshift-Operator geschrieben: $(1-B)\,Y_t = Y_t - Y_{t-1}$ und saisonal $(1-B^{12})\,Y_t = Y_t - Y_{t-12}$ (Vorlesung 3, Folien 37 und 41).

**Vorgehen (vor der Analyse festgelegt):**
1. Zuerst saisonal differenzieren ($D = 1$), danach regulär ($d = 1$) nur, falls noch ein Trend erkennbar ist; höchstens $d + D = 2$.
2. Hauptkriterium wie in Vorlesung 3, Folie 38: Mittelwert und Varianz wirken stabil, und die ACF fällt relativ schnell ab. Leitsatz: „So wenig differenzieren wie nötig“, da Überdifferenzierung künstliche Autokorrelation erzeugt.
3. Ergänzend (nicht Teil der Vorlesung): ADF-Test ($H_0$: nicht stationär) und KPSS-Test ($H_0$: stationär). Stationarität wird gestützt, wenn ADF ablehnt und KPSS nicht ablehnt. KPSS-p-Werte werden nur im Bereich 0,01–0,10 ausgegeben.
4. Die zusätzlich regulär differenzierte Reihe wird zum Vergleich mitberechnet, um eine Überdifferenzierung zu erkennen.

In [ ]:
reihen = {
    "Original $Y_t$": train,
    "$(1-B^{12})\\,Y_t$": arima.differenzieren(train, D=1),
    "$(1-B)(1-B^{12})\\,Y_t$": arima.differenzieren(train, d=1, D=1),
}
diff_tab = arima.differenzen_tabelle(reihen)
diff_tab.round(3)

In [ ]:
fig = arima.plot_differenzen(reihen)
speichern(fig, "09_differenzen");

**Ergebnis und Entscheidung:**
- **Originalreihe:** nicht stationär. Die ACF ist bei den saisonalen Lags hoch (Lag 12: 0,868) und fällt nur langsam ab; ADF lehnt nicht ab (p = 0,252), KPSS lehnt Stationarität ab (p ≤ 0,01).
- **Nach saisonaler Differenz $(1-B^{12})$:** Die Varianz sinkt von 52,59 auf 11,09, die Autokorrelation bei Lag 12 verschwindet (−0,005). Die Reihe schwankt um einen konstanten Mittelwert; die ACF liegt nach den ersten Lags überwiegend im Konfidenzband und zeigt keinen langsamen, linearen Abfall wie ein Random Walk (Vorlesung 3, Folie 12). ADF lehnt ab (p = 0,006), KPSS lehnt nicht ab (p ≥ 0,10): beide Tests stützen Stationarität.
- **Zusätzliche reguläre Differenz:** Die Autokorrelation bei Lag 1 wird stark negativ (−0,500) und die Varianz steigt auf 15,68. Das sind Anzeichen einer Überdifferenzierung (Vorlesung 3, Folie 38). Die Tests erkennen dies nicht (ADF p = 0,000), daher sind Grafik und ACF das Hauptkriterium.
- **Entscheidung:** $D = 1$, $d = 0$. Da $d + D = 1$, wird ein konstanter Term in das Modell aufgenommen und seine Signifikanz in Abschnitt 5 geprüft.
- Die verbleibende kurzfristige Autokorrelation der saisonal differenzierten Reihe wird in Abschnitt 3 über AR- und MA-Terme modelliert.

## 3. ACF und PACF der saisonal differenzierten Reihe

Grundlage ist $(1-B^{12})\,Y_t$ aus Abschnitt 2. Faustregeln aus Vorlesung 3: Die Ordnung $p$ ist der letzte signifikante Lag der PACF (Folie 26), die Ordnung $q$ der letzte signifikante Lag der ACF (Folie 30). Für den saisonalen Teil werden dieselben Regeln auf die Lags 12, 24 und 36 angewendet ($P$ aus der PACF, $Q$ aus der ACF). „Signifikant“ heißt: außerhalb des 95%-Bands.

In [ ]:
x = arima.differenzieren(train, D=1)
fig = arima.plot_acf_pacf(x, titel="$(1-B^{12})\\,Y_t$")
speichern(fig, "10_acf_pacf");

In [ ]:
acf_tab = arima.acf_pacf_tabelle(x)
print("Signifikante ACF-Lags: ", list(acf_tab.index[acf_tab["ACF signifikant"]]))
print("Signifikante PACF-Lags:", list(acf_tab.index[acf_tab["PACF signifikant"]]))
acf_tab.loc[[1, 2, 3, 4, 5, 6, 12, 24, 36]].round(3)

**Ergebnis: manuelles Modell**
- **Nicht-saisonal:** Die ACF ist bei fast allen Lags von 1 bis 11 signifikant und fällt allmählich ab, ohne abzubrechen → kein reiner MA-Prozess, $q = 0$. Die PACF ist bei Lag 1, 2 und 3 deutlich signifikant (0,293; 0,227; 0,422), bei Lag 4 nicht (0,097); Lag 5 überschreitet das Band nur knapp (0,139) und wird als Einzelwert nicht modelliert → $p = 3$ (Folie 26; kleinere Ordnungen bevorzugt, Folie 33).
- **Saisonal:** Die ACF ist bei Lag 12, 24 und 36 nicht signifikant → $Q = 0$. Die PACF ist bei Lag 12 und 24 signifikant (−0,364; −0,241), bei Lag 36 nicht (−0,055) → $P = 2$.
- **Manuelles Modell:** SARIMA(3,0,0)(2,1,0)$_{12}$ mit Konstante.
- **Einschränkungen:** Einzelne signifikante Werte bei höheren Lags (ACF: 14, 30; PACF: 14, 15, 16, 26, 34, 35) werden nicht modelliert; bei 36 Lags und 5 % Niveau sind je Funktion etwa 36 · 0,05 ≈ 1,8 zufällige Überschreitungen zu erwarten. Die abfallenden negativen PACF-Werte bei Lag 12 und 24 sind auch mit einem saisonalen MA-Term ($Q = 1$) vereinbar. Da die Bestimmung allein aus ACF und PACF schwierig ist (Folie 31), entscheidet in Abschnitt 4 die AIC-Suche über $P, Q \in \{0, 1, 2\}$.
- Das manuelle Modell liegt mit $p = 3$ außerhalb des Suchgitters ($p \le 2$) und wird in Abschnitt 4 zusätzlich geschätzt und per AIC verglichen (gleiches $d$, $D$ und gleiche Konstante).

## 4. Modellauswahl per AIC (Software)

Da die Bestimmung allein aus ACF und PACF schwierig ist (Vorlesung 3, Folie 31), werden die Ordnungen zusätzlich per Software gewählt (Folie 33). Kriterium ist das AIC (nicht Teil der Vorlesung):

$$\mathrm{AIC} = -2 \ln L + 2k$$

mit der maximierten Likelihood $L$ und der Anzahl geschätzter Parameter $k$ (einschließlich Konstante und $\sigma^2$). Kleiner ist besser; zusätzliche Parameter werden bestraft.

**Vorgehen (vor der Analyse festgelegt):**
- Gitter: $p, q, P, Q \in \{0, 1, 2\}$ (81 Modelle), fest: $d = 0$, $D = 1$, mit Konstante. Das AIC ist nur bei gleichem $d$, $D$ und gleicher Konstante vergleichbar.
- Das manuelle Modell aus Abschnitt 3 wird zusätzlich geschätzt.
- Kandidaten: das manuelle Modell, das Modell mit dem kleinsten AIC und das einfachste Modell (wenigste Parameter) mit ΔAIC < 2 zum besten Modell (gängige Faustregel; „eher kleinere Werte“, Folie 33).
- Die endgültige Wahl erfolgt in Abschnitt 5 nach der Residuendiagnose.

In [ ]:
import statsmodels
print("statsmodels", statsmodels.__version__)

gitter = arima.aic_gitter(train, d=0, D=1)   # dauert 1–3 Minuten
print(f"Geschätzte Modelle: {len(gitter)}, davon nicht konvergiert: {(~gitter['konvergiert']).sum()}")
gitter.head(10).round(2)

In [ ]:
manuell = arima.anpassen(train, (3, 0, 0), (2, 1, 0))
print(f"Manuelles Modell {arima.modell_name((3, 0, 0), (2, 1, 0))}: "
      f"AIC = {manuell.aic:.2f}, Parameter = {len(manuell.params)}, "
      f"ΔAIC zum besten Gittermodell = {manuell.aic - gitter['AIC'].min():.2f}, "
      f"konvergiert: {manuell.mle_retvals['converged']}")

In [ ]:
bester = gitter.iloc[0]
einfachster = gitter[gitter["ΔAIC"] < 2].sort_values(["Parameter", "AIC"]).iloc[0]
print(f"Bestes Modell (AIC):            {bester['Modell']} – {bester['Parameter']} Parameter, AIC = {bester['AIC']:.2f}")
print(f"Einfachstes Modell mit ΔAIC < 2: {einfachster['Modell']} – {einfachster['Parameter']} Parameter, ΔAIC = {einfachster['ΔAIC']:.2f}")

**Ergebnis: Kandidaten für Abschnitt 5**
- Alle 81 Gittermodelle wurden geschätzt; keines blieb ohne Konvergenz (Ausgabe oben).
- **Kleinstes AIC:** SARIMA(2,0,2)(2,1,2)$_{12}$ mit 10 Parametern (AIC = 1644,33). Alle vier Ordnungen liegen am oberen Rand des Gitters. Das kann auf eine reichhaltige Abhängigkeitsstruktur, aber auch auf überflüssige, sich gegenseitig aufhebende AR- und MA-Terme hinweisen; dies wird in Abschnitt 5 anhand der Koeffizienten geprüft.
- **Einfachstes Modell mit ΔAIC < 2:** SARIMA(1,0,1)(2,1,2)$_{12}$ mit 8 Parametern (ΔAIC = 1,93).
- **Manuelles Modell** SARIMA(3,0,0)(2,1,0)$_{12}$: AIC = 1685,56, ΔAIC = 41,23 und damit deutlich schlechter. Das bestätigt, dass die Bestimmung allein aus ACF und PACF schwierig ist (Vorlesung 3, Folie 31). Es bleibt als Vergleichsmodell im Kandidatenkreis.
- **Kandidaten** (jeweils mit Konstante): (1) SARIMA(1,0,1)(2,1,2)$_{12}$, (2) SARIMA(2,0,2)(2,1,2)$_{12}$, (3) SARIMA(3,0,0)(2,1,0)$_{12}$.
- Die Gittergrenze (Ordnung ≤ 2) wurde vorab festgelegt und nicht nachträglich erweitert.

## 5. Anpassung und Residuendiagnose (Leitfaden Punkt 6)

Die drei Kandidaten werden per Maximum Likelihood auf den Trainingsdaten geschätzt (wie in der Beispiel-Hausarbeit, Kap. 5). Bezeichnungen: $\varphi_i$, $\theta_j$ für die nicht-saisonalen AR- und MA-Koeffizienten, $\Phi_i$, $\Theta_j$ für die saisonalen, $c$ für die Konstante. $\varphi$ entspricht $\alpha$ und $\theta$ entspricht $\beta$ aus Vorlesung 3 (Folien 23 und 27); die Umbenennung vermeidet eine Verwechslung mit den Holt-Winters-Parametern.

Ein geeignetes Modell hinterlässt Residuen, die wie White Noise aussehen: konstanter Mittelwert, konstante Varianz, keine Autokorrelation (Vorlesung 3, Folie 7; Vorlesung 1, Folie 27). Geprüft werden:
- Verlauf und ACF der Residuen,
- Ljung-Box-Test bei Lag 12 und 24 ($H_0$: keine Autokorrelation; nicht Teil der Vorlesung), Freiheitsgrade um $p + q + P + Q$ reduziert,
- Histogramm und Q-Q-Plot (wie Beispiel-Hausarbeit, Kap. 6), ergänzend Jarque-Bera ($H_0$: Normalverteilung).

Die ersten $d + 12D = 12$ Residuen werden nicht verwendet, da sie durch die saisonale Differenz nicht aussagekräftig sind.

In [ ]:
kandidaten = {
    "K1 " + arima.modell_name((1, 0, 1), (2, 1, 2)): arima.anpassen(train, (1, 0, 1), (2, 1, 2)),
    "K2 " + arima.modell_name((2, 0, 2), (2, 1, 2)): arima.anpassen(train, (2, 0, 2), (2, 1, 2)),
    "K3 " + arima.modell_name((3, 0, 0), (2, 1, 0)): arima.anpassen(train, (3, 0, 0), (2, 1, 0)),
}
for name, res in kandidaten.items():
    print(name)
    display(arima.koeffizienten(res).round(4))
    display(arima.ljung_box(res).round(4))

In [ ]:
vergleich = arima.kandidaten_vergleich(kandidaten)
vergleich.round(4)

In [ ]:
for i, (name, res) in enumerate(kandidaten.items(), start=1):
    fig = arima.plot_residuen(res, titel=name)
    speichern(fig, f"11_residuen_k{i}")

**Ergebnis der Diagnose:**
- **Koeffizienten:** In K1 sind alle ARMA-Koeffizienten signifikant (p < 0,05); $\varphi_1$ liegt nahe 1 (≈ 0,99), kurzfristige Abweichungen bleiben also lange bestehen, $\theta_1$ (≈ −0,74) wirkt dem teilweise entgegen. In K2 sind $\varphi_1$ und $\theta_1$ nicht signifikant (p ≈ 0,22 bzw. 0,21): Das Modell am Gitterrand enthält überflüssige Parameter. K3 hat nur signifikante Koeffizienten, aber ein um 41,24 höheres AIC und die größte Residuenvarianz ($\sigma^2$ = 6,32).
- **Konstante:** In allen drei Modellen nicht signifikant (p ≈ 0,59; 0,60; 0,71). Sie bleibt gemäß der Vorab-Regel im Modell; eine signifikante Drift liegt nicht vor.
- **Autokorrelation der Residuen:** Der Ljung-Box-Test lehnt bei allen drei Kandidaten bei Lag 12 und 24 ab (p < 0,001). Die Residuen sind daher nicht vollständig White Noise; die ACF zeigt einzelne signifikante Lags (z. B. Lag 3 und 14). Eine mögliche Ursache (z. B. Kalendereffekte im Frühjahr) ist **nicht geprüft**.
- **Normalverteilung:** Jarque-Bera lehnt nicht ab (p = 0,098; 0,082; 0,867), der Q-Q-Plot ist bis auf leichte Abweichungen an den Rändern linear. Die Normalverteilungsannahme für Prognoseintervalle ist vertretbar.

## 6. Festlegung vor dem Test

**Vorab festgelegte Regel:** Gewählt wird der Kandidat mit dem kleinsten AIC, dessen Residuen den Ljung-Box-Test bei Lag 12 und 24 bestehen (p > 0,05). Liegen zwei Modelle weniger als 2 AIC-Punkte auseinander, wird das einfachere gewählt. Besteht kein Kandidat, wird nach AIC gewählt und die verbleibende Autokorrelation als Einschränkung berichtet.

In [ ]:
lb_ok = vergleich[["Ljung-Box p (Lag 12)", "Ljung-Box p (Lag 24)"]].min(axis=1) > 0.05
print("Kandidaten, die Ljung-Box bei Lag 12 und 24 bestehen:", list(vergleich.index[lb_ok]))
print(f"AIC-Abstand K1 – K2: {vergleich['AIC'].iloc[0] - vergleich['AIC'].iloc[1]:.2f}")

**Entscheidung:** Kein Kandidat besteht den Ljung-Box-Test. Nach AIC liegen K2 und K1 weniger als 2 Punkte auseinander; gemäß der Regel wird das einfachere Modell gewählt. Wörtlich genommen hätte der letzte Teil der Regel („nach AIC wählen“) K2 ergeben. Für K1 sprechen zusätzlich die zwei nicht signifikanten Koeffizienten in K2. Die Wahl beruht ausschließlich auf den Trainingsdaten.

**Endgültiges Modell:** SARIMA(1,0,1)(2,1,2)$_{12}$ mit Konstante, allgemein:

$$(1-\varphi_1 B)(1-\Phi_1 B^{12}-\Phi_2 B^{24})(1-B^{12})\,Y_t = c + (1+\theta_1 B)(1+\Theta_1 B^{12}+\Theta_2 B^{24})\,\varepsilon_t,\quad \varepsilon_t \sim \mathrm{WN}(0,\sigma^2)$$

$B$ ist der Backshift-Operator ($B^k Y_t = Y_{t-k}$, Vorlesung 3, Folie 37). Die geschätzte Gleichung steht in der nächsten Zelle. K2 und K3 werden in Abschnitt 7 nur zur Information mitprognostiziert.

In [ ]:
from IPython.display import Math, display

final_name = list(kandidaten)[0]          # K1 SARIMA(1,0,1)(2,1,2)[12]
final = kandidaten[final_name]
print("Endgültiges Modell:", final_name)
display(Math(arima.modellgleichung(final)))
print(f"σ² = {final.params['sigma2']:.3f}")

## 7. Prognose auf der Testmenge (Leitfaden Punkt 7)

Die Testmenge wird hier **zum ersten und einzigen Mal** verwendet. Das in Abschnitt 6 festgelegte Modell (auf 368 Trainingsmonaten geschätzt) prognostiziert alle 24 Testmonate auf einmal; es wird nicht neu geschätzt. ARIMA modelliert Veränderungen, prognostiziert aber Niveaus; die Rückintegration der saisonalen Differenz übernimmt die Software (Vorlesung 3, Folie 42).

**Prognoseintervall (95 %):** $\hat y_{T+h} \pm 1{,}96\,\sigma_h$, wobei $\sigma_h$ mit dem Horizont $h$ wächst. Annahmen: normalverteilte, unabhängige Fehler und bekannte Parameter. Da die Residuen laut Ljung-Box nicht vollständig unabhängig sind (Abschnitt 5), können die Intervalle zu schmal sein.

Bewertung mit denselben Metriken wie Holt-Winters und die Baseline (`src/evaluation.py`). K2 und K3 werden **nur zur Information** gezeigt; die Ergebnisse auf der Testmenge ändern die Modellwahl nicht.

In [ ]:
from src.evaluation import evaluate, rmse, seasonal_naive
from src import holtwinters as hw

train, test = train_test_split(s)   # erst jetzt wird die Testmenge geladen
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")
print(f"Test:     {test.index[0]:%m/%Y}–{test.index[-1]:%m/%Y} ({len(test)} Monate)")

In [ ]:
punkt, ki = arima.prognose(final, len(test))
assert punkt.index.equals(test.index)

ergebnis = evaluate(test, punkt)
print("SARIMA (endgültig):", {k: round(v, 2) for k, v in ergebnis.items()})
print("Seasonal Naive:    ", {k: round(v, 2) for k, v in evaluate(test, seasonal_naive(train)).items()})
print(f"Abdeckung des 95%-Intervalls: {arima.abdeckung(test, ki):.1%} "
      f"({int(round(arima.abdeckung(test, ki) * len(test)))} von {len(test)} Monaten)")
print(f"RMSE 1. Testjahr (09/2024–08/2025): {rmse(test.iloc[:12], punkt.iloc[:12]):.2f}")
print(f"RMSE 2. Testjahr (09/2025–08/2026): {rmse(test.iloc[12:], punkt.iloc[12:]):.2f}")
print(f"Mittlerer Fehler (ŷ − y): {(punkt - test).mean():.2f}, "
      f"Prognose zu hoch in {(punkt > test).sum()} von {len(test)} Monaten")

In [ ]:
fig = arima.plot_prognose(train, test, punkt, ki)
speichern(fig, "12_arima_prognose");

In [ ]:
fehler = hw.fehler_je_monat(test, {"SARIMA": punkt, "Seasonal Naive": seasonal_naive(train)})
fig = hw.plot_fehler(fehler)
speichern(fig, "13_arima_fehler");
fehler.round(2)

In [ ]:
zeilen = {}
for name, res in kandidaten.items():
    p, k = arima.prognose(res, len(test))
    zeilen[name + ("  (endgültig)" if name == final_name else "  (nur zur Info)")] = {
        **evaluate(test, p), "Abdeckung 95 %": arima.abdeckung(test, k)}
pd.DataFrame(zeilen).T.round(3)

**Ergebnis auf der Testmenge:**
- Das endgültige Modell SARIMA(1,0,1)(2,1,2)$_{12}$ erreicht RMSE = 3,33, MAE = 2,86 und MAPE = 3,02 %. Es liegt damit knapp hinter der Seasonal-Naive-Baseline (RMSE = 3,28) und deutlich hinter Holt-Winters (RMSE = 2,44, siehe `03_holtwinters.ipynb`).
- Die Prognose ist systematisch zu hoch: mittlerer Fehler +2,25, zu hoch in 19 von 24 Monaten. Im zweiten Testjahr ist der Fehler größer (RMSE 3,55 gegenüber 3,09).
- Das 95%-Prognoseintervall enthält 23 von 24 Testwerten (95,8 %). Trotz der Residuenautokorrelation sind die Intervalle hier nicht erkennbar zu schmal; bei 24 Monaten ist diese Aussage jedoch unsicher.
- **Nur zur Information:** K3 (manuelles Modell, schlechtestes AIC) erreicht auf der Testmenge RMSE = 2,41, K2 RMSE = 3,28. Eine gute Anpassung an die Trainingsdaten garantiert also keine gute Prognose. Die Modellwahl wird dadurch **nicht** geändert, da sie sonst auf der Testmenge beruhen würde.
- Mögliche Erklärung (**nicht geprüft**): Wegen $\varphi_1 \approx 0{,}99$ wirken Abweichungen vom Ende der Trainingsdaten über den gesamten Prognosehorizont fort.

## 8. Zusammenfassung

- **Identifikation (Trainingsdaten):** keine Box-Cox-Transformation (Varianz unabhängig vom Niveau), saisonale Differenz $D = 1$, keine reguläre Differenz ($d = 0$), Konstante im Modell.
- **Modellwahl:** AIC-Gitter über $p, q, P, Q \in \{0, 1, 2\}$ plus manuelles Modell; endgültig SARIMA(1,0,1)(2,1,2)$_{12}$, festgelegt **vor** der Prognose (Commit 2587682).
- **Diagnose:** Residuen annähernd normalverteilt, aber nicht vollständig frei von Autokorrelation (Ljung-Box).
- **Prognose:** RMSE 3,33 auf den 24 Testmonaten, schlechter als Holt-Winters (2,44) und knapp schlechter als Seasonal Naive (3,28); systematische Überschätzung.
- **Einschränkungen und Erweiterungen:** nur ein Testzeitraum; eine rollierende Kreuzvalidierung (Vorlesung 2, Folie 28) würde die Modellwahl robuster bewerten. Verbleibende Residuenautokorrelation könnte mit exogenen Kalendervariablen (SARIMAX) untersucht werden. Höhere Ordnungen wurden bewusst nicht betrachtet („eher kleinere Werte“, Vorlesung 3, Folie 33).